# 개별종목 조합I — LightGBM

`기본모델/04.LightGBM.ipynb`과 같은 `models.lightgbm.build_lightgbm_baseline`을 가져오고
조합I 피처를 주입합니다. 기본모델 코드는 `models/`에 한 번만 존재합니다.
후보·라벨·날짜 그룹 12폴드 실행은 모든 조합이 같은 공통 함수를 사용합니다.


In [1]:
# 1. 기본모델을 가져옵니다.
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from models.lightgbm import build_lightgbm_baseline  # noqa: E402

MODEL_NAME = 'LightGBM'
MODEL_BUILDER = build_lightgbm_baseline


In [2]:
# 2. 조합I의 피처 값만 지정합니다.
import json

COMBINATION = 'I'
FEATURE_COLUMNS = (
    'atr_ratio',
    'bb_bandwidth',
    'hv_regime',
    'five_day_return',
)

report_path = project_root / "reports" / "stock_feature_combinations.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
combination_report = report["combinations"][COMBINATION]
panel = combination_report["panel"]
print("학습 기간:", panel["first_date"], "~", panel["last_date"])
print("학습 행·종목:", panel["model_rows"], panel["stocks"])
print(f"조합{COMBINATION} 피처:", FEATURE_COLUMNS)

folds = pd.DataFrame(combination_report["outer_fold_results"])
model_folds = folds.loc[folds["model"].eq(MODEL_NAME)].reset_index(drop=True)
fold_columns = [
    "fold",
    "selected_class_weight",
    "train_dates",
    "valid_start",
    "valid_end",
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, fold_columns].round(4))

metric_columns = [
    "accuracy",
    "training_majority_baseline_accuracy",
    "accuracy_minus_training_majority_baseline",
    "macro_f1",
    "balanced_accuracy",
    "mcc",
    "pr_auc_macro_ovr",
    "down_recall",
    "core_harmonic_mean",
]
display(model_folds.loc[:, metric_columns].mean().to_frame("OOS 폴드 평균").round(4))

# 조합별 노트북이 중복 학습하지 않도록 실제 fit은 공통 실행기에서 한 번 수행합니다.
print("재실행 명령: python scripts/run_stock_model_experiment.py")


학습 기간: 20110127 ~ 20240822
학습 행·종목: 159936 157
조합I 피처: ('atr_ratio', 'bb_bandwidth', 'hv_regime', 'five_day_return')


,fold,selected_class_weight,train_dates,valid_start,valid_end,accuracy,training_majority_baseline_accuracy,accuracy_minus_training_majority_baseline,macro_f1,balanced_accuracy,mcc,pr_auc_macro_ovr,down_recall,core_harmonic_mean
0,1,balanced,750,20140217,20140514,0.5023,0.5012,0.0010,0.3317,0.3705,0.0932,0.3805,0.1245,0.2300
1,2,balanced,980,20150123,20150421,0.3853,0.3978,-0.0126,0.3489,0.3585,0.0476,0.3755,0.2008,0.2873
2,3,balanced,1210,20151228,20160328,0.3570,0.3762,-0.0192,0.3559,0.3562,0.0374,0.3682,0.3278,0.3464
3,4,balanced,1439,20161202,20170228,0.4551,0.4617,-0.0067,0.3668,0.3817,0.0933,0.3980,0.1728,0.2801
4,5,balanced,1669,20171113,20180207,0.4147,0.3901,0.0246,0.3806,0.3919,0.0980,0.3884,0.2610,0.3382
5,6,balanced,1899,20181024,20190118,0.4150,0.3725,0.0425,0.4139,0.4229,0.1366,0.4197,0.5111,0.4423
6,7,balanced,2129,20190930,20191224,0.4717,0.4781,-0.0065,0.3576,0.3795,0.0985,0.4017,0.1806,0.2870
7,8,balanced,2359,20200902,20201130,0.3949,0.3476,0.0473,0.3923,0.3965,0.0959,0.3957,0.4332,0.4060
8,9,balanced,2589,20210806,20211105,0.3954,0.3916,0.0038,0.3863,0.3960,0.0910,0.3892,0.3113,0.3602
9,10,balanced,2818,20220714,20221012,0.3399,0.3454,-0.0055,0.3400,0.3448,0.0177,0.3540,0.2585,0.3076


,OOS 폴드 평균
accuracy,0.4105
training_majority_baseline_accuracy,0.3969
accuracy_minus_training_majority_baseline,0.0136
macro_f1,0.3719
balanced_accuracy,0.3825
mcc,0.0835
pr_auc_macro_ovr,0.3889
down_recall,0.2876
core_harmonic_mean,0.3359


재실행 명령: python scripts/run_stock_model_experiment.py
